# Fundamentals of Data Science — Case Studies

**Student:** Annika Reddy  
**Roll Number:** 160124733153

This notebook presents exploratory analysis and predictive modelling for two datasets: online shopping baskets and student academic outcomes.

## 1. Dataset loading

The notebook loads the Market Basket Optimisation dataset for transaction analysis and the UCI Predict Students' Dropout and Academic Success dataset for student outcome analysis.

In [ ]:
%pip -q install pandas numpy matplotlib scikit-learn scipy requests
import io, zipfile, requests, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

In [ ]:
# Dataset 1: 7,501 transaction baskets, commonly used for market-basket analysis.
retail_url = "https://huggingface.co/spaces/Iris230/space_recommend/resolve/main/Market_Basket_Optimisation.csv"
retail_response = requests.get(retail_url, timeout=60)
retail_response.raise_for_status()
retail = pd.read_csv(io.BytesIO(retail_response.content), header=None, dtype=str, keep_default_na=False)
print("Retail dataset shape:", retail.shape)
display(retail.head())

# Dataset 2: official UCI student dropout/academic success dataset.
student_zip_url = "https://archive.ics.uci.edu/static/public/697/predict+students+dropout+and+academic+success.zip"
student_response = requests.get(student_zip_url, timeout=60)
student_response.raise_for_status()
with zipfile.ZipFile(io.BytesIO(student_response.content)) as z:
    csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
    print("CSV files in UCI archive:", csv_names)
    preferred = next((n for n in csv_names if n.lower().endswith("data.csv")), csv_names[0])
    with z.open(preferred) as f:
        students = pd.read_csv(f, sep=";", encoding="utf-8-sig")
students.columns = students.columns.str.strip()
print("Student dataset shape:", students.shape)
display(students.head())

# Case Study 1 — Online Retail
## 1. Clean baskets and explore product frequencies

In [ ]:
baskets = []
for _, row in retail.iterrows():
    items = [str(v).strip().lower() for v in row.tolist() if str(v).strip()]
    baskets.append(sorted(set(items)))
basket_sizes = np.array([len(b) for b in baskets])
products = sorted({item for basket in baskets for item in basket})
basket_matrix = np.zeros((len(baskets), len(products)), dtype=np.uint8)
product_to_col = {p:i for i,p in enumerate(products)}
for i, basket in enumerate(baskets):
    for item in basket:
        basket_matrix[i, product_to_col[item]] = 1
basket_df = pd.DataFrame(basket_matrix, columns=products)

print("Number of baskets:", len(baskets))
print("Distinct products:", len(products))
print("Total basket-product entries:", int(basket_matrix.sum()))
print("Basket size mean/std/median/min/max:",
      round(basket_sizes.mean(), 3), round(basket_sizes.std(), 3),
      float(np.median(basket_sizes)), int(basket_sizes.min()), int(basket_sizes.max()))

product_counts = basket_df.sum().sort_values(ascending=False)
product_percent = 100 * product_counts / len(basket_df)
display(pd.DataFrame({"Basket count": product_counts, "Basket share (%)": product_percent.round(2)}).head(15))
product_percent.head(15).sort_values().plot(kind="barh", figsize=(8,5), title="Top 15 Products by Basket Share")
plt.xlabel("Baskets containing product (%)"); plt.tight_layout(); plt.show()

pd.Series(basket_sizes).value_counts().sort_index().plot(kind="bar", figsize=(8,4), title="Basket-size Distribution")
plt.xlabel("Number of products in basket"); plt.ylabel("Number of baskets"); plt.tight_layout(); plt.show()

## 2. Association rules (pairwise support, confidence, lift)
For a pair A → B: support = P(A and B), confidence = P(B given A), lift = confidence / P(B). These indicate co-occurrence, not causation.

In [ ]:
n = len(basket_df)
min_support_count = int(np.ceil(0.01*n))
eligible = product_counts[product_counts >= min_support_count].index.tolist()
assoc = basket_df[eligible]
rows = []
for a in eligible:
    a_count = int(assoc[a].sum())
    for b in eligible:
        if a == b: 
            continue
        b_count = int(assoc[b].sum())
        joint = int(((assoc[a] == 1) & (assoc[b] == 1)).sum())
        if joint >= 75 and a_count and b_count:
            confidence = joint/a_count
            rows.append((a,b,joint,joint/n,confidence,confidence/(b_count/n)))
rules = pd.DataFrame(rows, columns=["If basket has","Then also","Joint baskets","Support","Confidence","Lift"])
display(rules.sort_values(["Lift","Confidence"], ascending=False).head(20).round(4))

## 3. PCA and clustering
High-dimensional binary baskets can be difficult to separate into clear clusters. The following compares K-means, hierarchical, spectral clustering and DBSCAN on a reproducible sample. Silhouette and Davies–Bouldin scores are descriptive metrics; their values depend on settings.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, pairwise_distances

rng = np.random.RandomState(RANDOM_STATE)
sample_n = min(2000, len(basket_df))
idx = rng.choice(len(basket_df), sample_n, replace=False)
X = basket_df.iloc[idx].to_numpy()
pca = PCA(random_state=RANDOM_STATE)
Xp = pca.fit_transform(X)
print("PCA first two components variance (%):",
      (pca.explained_variance_ratio_[:2]*100).round(2))
print("Cumulative variance first 10 components (%):",
      round(pca.explained_variance_ratio_[:10].sum()*100, 2))
plt.figure(figsize=(7,4))
plt.bar(range(1, min(10, len(pca.explained_variance_ratio_))+1),
        pca.explained_variance_ratio_[:10]*100)
plt.xlabel("Principal component"); plt.ylabel("Variance explained (%)")
plt.title("PCA Scree Plot"); plt.tight_layout(); plt.show()

km_results=[]
for k in range(2,9):
    labels=KMeans(n_clusters=k,n_init=10,random_state=RANDOM_STATE).fit_predict(X)
    km_results.append((k, silhouette_score(X,labels), davies_bouldin_score(X,labels)))
km_df=pd.DataFrame(km_results,columns=["k","Silhouette","Davies-Bouldin"])
display(km_df.round(4))
km_df.set_index("k")[["Silhouette","Davies-Bouldin"]].plot(marker="o",subplots=True,figsize=(7,6),title="K-means validation scores")
plt.tight_layout(); plt.show()

cluster_models = {
    "K-means (k=5)": KMeans(n_clusters=5,n_init=10,random_state=RANDOM_STATE),
    "Hierarchical (complete, k=5)": AgglomerativeClustering(n_clusters=5,linkage="complete"),
    "Spectral (k=5)": SpectralClustering(n_clusters=5,affinity="nearest_neighbors",
                                          n_neighbors=min(15,len(X)-1),random_state=RANDOM_STATE)
}
cluster_rows=[]
for name, model in cluster_models.items():
    labels=model.fit_predict(X)
    cluster_rows.append((name, len(set(labels)), silhouette_score(X,labels),
                         davies_bouldin_score(X,labels)))
display(pd.DataFrame(cluster_rows,columns=["Method","Clusters","Silhouette","Davies-Bouldin"]).round(4))

# DBSCAN with Jaccard distance on binary basket features
D=pairwise_distances(X,metric="jaccard")
db_labels=DBSCAN(eps=0.5,min_samples=5,metric="precomputed").fit_predict(D)
mask=db_labels!=-1
print("DBSCAN clusters excluding noise:", len(set(db_labels[mask])))
print("DBSCAN noise points:", int((~mask).sum()))
if mask.sum()>1 and len(set(db_labels[mask]))>1:
    print("DBSCAN silhouette excluding noise:", round(silhouette_score(X[mask],db_labels[mask]),4))

# Case Study 4 — Student Data and Performance
## 1. Data checks and outcome distribution

In [ ]:
print("Rows, columns:", students.shape)
print("Missing values:", int(students.isna().sum().sum()))
print("Duplicate rows:", int(students.duplicated().sum()))
target_col = "Target"
display(students[target_col].value_counts().rename_axis("Outcome").to_frame("Count"))
(students[target_col].value_counts(normalize=True)*100).plot(kind="bar",figsize=(7,4),
    title="Final Student Outcome Distribution", ylabel="Percentage (%)")
plt.tight_layout(); plt.show()

## 2. Exploratory analysis and statistical hypothesis test
The Kruskal–Wallis test compares grade distributions across the three outcome groups. A small p-value suggests at least one group differs; it does not by itself identify which groups differ.

In [ ]:
from scipy.stats import kruskal
grade_col = "Curricular units 1st sem (grade)"
if grade_col in students.columns:
    display(students.groupby(target_col)[grade_col].agg(["count","mean","median","std"]).round(3))
    students.boxplot(column=grade_col, by=target_col, figsize=(8,5))
    plt.title("First-semester Grade by Final Outcome"); plt.suptitle("")
    plt.xlabel("Final outcome"); plt.ylabel("First-semester grade")
    plt.tight_layout(); plt.show()
    groups=[g[grade_col].dropna().values for _,g in students.groupby(target_col)]
    H,p=kruskal(*groups)
    n=sum(len(g) for g in groups); k=len(groups)
    epsilon_sq=max(0,(H-k+1)/(n-k))
    print(f"Kruskal-Wallis H={H:.4f}, p-value={p:.6g}, epsilon-squared={epsilon_sq:.4f}")

## 3. Logistic regression: standard vs class-weighted
To reduce target leakage for an early prediction task, this comparison uses first-semester and admission-time information and excludes second-semester features. Results may differ from the report if its split, selected features, or hyperparameters differed.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix

y=students[target_col].astype(str)
X=students.drop(columns=[target_col]).copy()
second_sem_cols=[c for c in X.columns if "2nd sem" in c.lower() or "2nd semester" in c.lower()]
X=X.drop(columns=second_sem_cols,errors="ignore")
cat_cols=X.select_dtypes(include=["object","category"]).columns.tolist()
num_cols=[c for c in X.columns if c not in cat_cols]
preprocessor=ColumnTransformer([
    ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num_cols),
    ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat_cols)
])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=RANDOM_STATE,stratify=y)

models={
    "Standard logistic regression": LogisticRegression(max_iter=3000,C=0.5,random_state=RANDOM_STATE),
    "Class-weighted logistic regression": LogisticRegression(max_iter=3000,C=0.5,class_weight="balanced",random_state=RANDOM_STATE)
}
results={}
for name, clf in models.items():
    model=Pipeline([("preprocess",preprocessor),("classifier",clf)])
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    results[name]=(model,pred)
    print("\n",name)
    print("Accuracy:",round(accuracy_score(y_test,pred),4))
    print("Balanced accuracy:",round(balanced_accuracy_score(y_test,pred),4))
    print("Macro F1:",round(f1_score(y_test,pred,average="macro"),4))
    print(classification_report(y_test,pred,zero_division=0))

In [ ]:
for name,(model,pred) in results.items():
    labels=model.named_steps["classifier"].classes_
    cm=confusion_matrix(y_test,pred,labels=labels)
    fig,ax=plt.subplots(figsize=(5,4))
    ax.imshow(cm)
    ax.set_title("Confusion Matrix — "+name)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_xticks(range(len(labels)),labels=labels,rotation=25)
    ax.set_yticks(range(len(labels)),labels=labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j,i,str(cm[i,j]),ha="center",va="center")
    plt.tight_layout(); plt.show()